In [1]:
#Imports and load clean data
import pandas as pd
import numpy as np
import os

DATA_DIR = "C:/Users/Aishwarya/Desktop/job-market-project/output/"

postings = pd.read_csv(DATA_DIR + "postings_clean.csv", low_memory=False)
job_skills = pd.read_csv(DATA_DIR + "job_skills_named.csv")
job_industries = pd.read_csv(DATA_DIR + "job_industries_named.csv")
benefits = pd.read_csv(DATA_DIR + "benefits_clean.csv")
companies = pd.read_csv(DATA_DIR + "companies_clean.csv")

print(f"Postings: {len(postings):,} rows")
print(f"Skills: {len(job_skills):,} rows")
print(f"Industries: {len(job_industries):,} rows")

Postings: 123,849 rows
Skills: 213,768 rows
Industries: 164,808 rows


In [2]:
#Most demanded skills
# Count how many jobs require each skill
top_skills = (
    job_skills.groupby('skill_abr')['job_id']
    .count()
    .reset_index()
    .rename(columns={'job_id': 'job_count'})
    .sort_values('job_count', ascending=False)
    .head(20)
)

print(top_skills)
top_skills.to_csv(DATA_DIR + "analytics_top_skills.csv", index=False)


   skill_abr  job_count
16        IT      26137
29      SALE      22475
18      MGMT      20861
19      MNFC      18185
14      HCPR      17369
5         BD      14290
11       ENG      13009
21      OTHR      12608
12       FIN       8540
20      MRKT       5525
0       ACCT       5461
1        ADM       4860
7       CUST       4292
25      PRJM       3997
3       ANLS       3858
28      RSCH       2986
15        HR       2647
17       LGL       2371
6       CNSL       2338
10       EDU       2290


In [4]:
#Salary job title
# Average salary for top 20 most common job titles
salary_by_title = (
    postings[postings['clean_salary'].notna()]
    .groupby('title')['clean_salary']
    .agg(['mean', 'median', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_salary', 'median': 'med_salary', 'count': 'job_count'})
    .query('job_count >= 10')  # only titles with enough data
    .sort_values('avg_salary', ascending=False)
    .head(20)
)

print(salary_by_title)
salary_by_title.to_csv(DATA_DIR + "analytics_salary_by_title.csv", index=False)

                                         title     avg_salary  med_salary  \
3589                   Chief Financial Officer  247500.000000    250000.0   
15164                   Primary Care Physician  213750.000000    225000.0   
9346                             Head of Sales  211900.000000    218750.0   
17446  Remote Licensed Mental Health Counselor  188334.545455    187200.0   
15317              Principal Software Engineer  186195.000000    199825.0   
11434         Licensed Mental Health Therapist  180856.000000    178880.0   
20262         Senior Site Reliability Engineer  175392.000000    166480.0   
11593                     Litigation Associate  169500.000000    146250.0   
20029                   Senior Product Manager  168786.230769    165000.0   
20278                 Senior Software Engineer  165161.419643    153316.0   
11597                      Litigation Attorney  159750.000000    150000.0   
20768                Site Reliability Engineer  156943.680000    145600.0   

In [6]:
#Salary by city
# List of non-city values to exclude
non_cities = [
    'United States', 'California', 'Illinois', 'Texas', 'Florida', 'New York',
    'Washington', 'Georgia', 'Ohio', 'Pennsylvania', 'Massachusetts', 'Virginia',
    'New Jersey', 'Colorado', 'Arizona', 'Michigan', 'North Carolina',
    'Washington DC-Baltimore Area', 'New York City Metropolitan Area',
    'San Francisco Bay Area', 'Los Angeles Metropolitan Area',
    'Greater Chicago Area', 'Greater Seattle Area', 'Greater Boston Area',
    'Greater New York City Area', 'Greater Los Angeles Area'
]

salary_by_city = (
    postings[
        postings['clean_salary'].notna() &
        postings['city'].notna() &
        ~postings['city'].isin(non_cities) &
        ~postings['city'].str.contains('Area|Metro|Region|Greater', na=False)
    ]
    .groupby('city')['clean_salary']
    .agg(['mean', 'median', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_salary', 'median': 'med_salary', 'count': 'job_count'})
    .query('job_count >= 50')
    .sort_values('avg_salary', ascending=False)
    .head(20)
)

print(salary_by_city)
salary_by_city.to_csv(DATA_DIR + "analytics_salary_by_city.csv", index=False)

                    city     avg_salary  med_salary  job_count
2147       Mountain View  160321.611864    161200.0         59
3128           Sunnyvale  150945.092000    151375.0        100
2450           Palo Alto  146503.194179    137717.5         67
2839         Santa Clara  143658.164333    132250.0         90
2816            San Jose  139854.468067    136525.0        238
2814       San Francisco  138230.107351    130770.0        404
215             Bellevue  129559.642576    119953.6         99
2890             Seattle  126693.124188    125000.0        431
1961              McLean  126424.337500    129500.0         72
1543         Jersey City  124625.828767    130000.0         73
2666             Redmond  121295.258065    115350.0         93
313               Boston  114237.926751    104000.0        334
2847        Santa Monica  113060.332836    112500.0         67
2550               Plano  111984.569444    113750.0         72
1145             Fremont  110984.779712    104999.2    

In [7]:
#Remote vs onsite
remote_counts = (
    postings.groupby('is_remote')['job_id']
    .count()
    .reset_index()
    .rename(columns={'job_id': 'job_count'})
)
remote_counts['label'] = remote_counts['is_remote'].map({0: 'Onsite', 1: 'Remote'})
remote_counts['percentage'] = (remote_counts['job_count'] / remote_counts['job_count'].sum() * 100).round(1)

print(remote_counts)
remote_counts.to_csv(DATA_DIR + "analytics_remote_vs_onsite.csv", index=False)

   is_remote  job_count   label  percentage
0          0     108489  Onsite        87.6
1          1      15360  Remote        12.4


In [9]:
#Hiring by city
jobs_by_city = (
    postings[
        postings['city'].notna() &
        ~postings['city'].isin(non_cities) &
        ~postings['city'].str.contains('Area|Metro|Region|Greater|County', na=False)
    ]
    .groupby('city')['job_id']
    .count()
    .reset_index()
    .rename(columns={'job_id': 'job_count'})
    .sort_values('job_count', ascending=False)
    .head(20)
)

print(jobs_by_city)
jobs_by_city.to_csv(DATA_DIR + "analytics_jobs_by_city.csv", index=False)

               city  job_count
936         Chicago       1836
2456        Houston       1776
1257         Dallas       1394
198         Atlanta       1369
222          Austin       1325
528          Boston       1202
3050    Los Angeles       1100
895       Charlotte       1086
4165        Phoenix       1062
4659  San Francisco        887
4776        Seattle        819
1344         Denver        796
4656      San Diego        796
4650    San Antonio        769
1082       Columbus        733
4161   Philadelphia        732
5220          Tampa        667
3342          Miami        656
4475       Richmond        597
3971        Orlando        593


In [10]:
#Jobs by experience level
jobs_by_experience = (
    postings[postings['formatted_experience_level'].notna()]
    .groupby('formatted_experience_level')['job_id']
    .count()
    .reset_index()
    .rename(columns={'job_id': 'job_count'})
    .sort_values('job_count', ascending=False)
)

print(jobs_by_experience)
jobs_by_experience.to_csv(DATA_DIR + "analytics_jobs_by_experience.csv", index=False)

  formatted_experience_level  job_count
5           Mid-Senior level      41489
2                Entry level      36708
0                  Associate       9826
1                   Director       3746
4                 Internship       1449
3                  Executive       1222


In [11]:
#Top industries hiring
top_industries = (
    job_industries[job_industries['industry_name'].notna()]
    .groupby('industry_name')['job_id']
    .count()
    .reset_index()
    .rename(columns={'job_id': 'job_count'})
    .sort_values('job_count', ascending=False)
    .head(20)
)

print(top_industries)
top_industries.to_csv(DATA_DIR + "analytics_top_industries.csv", index=False)

                          industry_name  job_count
148           Hospitals and Health Care      18326
284                              Retail      11033
156       IT Services and IT Consulting      10396
324             Staffing and Recruiting       9005
115                  Financial Services       8535
315                Software Development       5091
195                       Manufacturing       3689
69                         Construction       3445
30                              Banking       2923
169                           Insurance       2673
249        Pharmaceutical Manufacturing       2469
146                         Hospitality       2455
331                  Telecommunications       2433
270                         Real Estate       2326
166  Industrial Machinery Manufacturing       2143
225            Non-profit Organizations       2063
34               Biotechnology Research       2005
3                            Accounting       1968
215         Motor Vehicle Manuf

In [12]:
#Most common benefits
top_benefits = (
    benefits.groupby('type')['job_id']
    .count()
    .reset_index()
    .rename(columns={'job_id': 'job_count'})
    .sort_values('job_count', ascending=False)
)

print(top_benefits)
top_benefits.to_csv(DATA_DIR + "analytics_top_benefits.csv", index=False)

                       type  job_count
0                    401(k)      24231
5         Medical insurance       9873
11         Vision insurance       9309
4      Disability insurance       7930
3          Dental insurance       6868
10       Tuition assistance       2614
2         Commuter benefits       2226
6      Paid maternity leave       1808
7      Paid paternity leave       1540
8              Pension plan        906
9   Student loan assistance        365
1        Child care support        273
